# Multi-Step Funnel & Friction Analysis

## 1. Business Objective

Analyze the SaaS user journey across five core funnel stages to identify conversion bottlenecks and compare user behaviour across Desktop and Mobile devices.

## 2. Analytical Objectives

- Clean and prepare raw clickstream event data.
- Deduplicate users at each funnel stage.
- Calculate distinct users reaching each stage.
- Measure cumulative and step-over-step conversion rates.
- Calculate absolute and percentage drop-offs.
- Compare funnel performance across Desktop and Mobile.
- Visualize the conversion funnel and identify major friction points.

## 3. Tools & Libraries

Python, pandas, NumPy, SciPy, Matplotlib, Seaborn and Plotly.

In [1]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
from scipy import stats

print("Python environment connected successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Python environment connected successfully!
Pandas version: 3.1.1
NumPy version: 2.5.5


# Multi-Step Funnel & Friction Analysis

## 1. Data Loading and Initial Inspection

This analysis uses the clickstream dataset generated by the
assignment's master notebook. The objective is to reproduce
the funnel analysis independently using pandas.

In [2]:
# Load the original clickstream dataset
df = pd.read_csv("../data/clickstream_events.csv")

# Inspect the dataset
print(f"Dataset shape: {df.shape}")
print(f"Unique users: {df['user_id'].nunique():,}")

df.head()

Dataset shape: (26061, 3)
Unique users: 10,000


,user_id,device,event_name
0,CUST_00001,desktop,1_Homepage
1,CUST_00002,mobile,1_Homepage
2,CUST_00002,mobile,2_Product_View
3,CUST_00002,mobile,3_Add_to_Cart
4,CUST_00002,mobile,4_Checkout_Start


## 2. Data Quality Assessment

Inspect column data types, missing values, duplicate records,
and distinct values before performing funnel calculations.

In [3]:
# Dataset structure
df.info()

# Missing values
print("\nMissing values:")
display(df.isna().sum())

# Exact duplicate rows
print("\nExact duplicate rows:", df.duplicated().sum())

# Unique values in categorical columns
print("\nDevice categories:")
print(df['device'].value_counts())

print("\nFunnel events:")
print(df['event_name'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 26061 entries, 0 to 26060
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   user_id     26061 non-null  str  
 1   device      26061 non-null  str  
 2   event_name  26061 non-null  str  
dtypes: str(3)
memory usage: 610.9 KB

Missing values:


user_id       0
device        0
event_name    0
dtype: int64


Exact duplicate rows: 3000

Device categories:
device
desktop    14259
mobile     11802
Name: count, dtype: int64

Funnel events:
event_name
1_Homepage           11273
2_Product_View        7579
3_Add_to_Cart         3874
4_Checkout_Start      1869
5_Signup_Complete     1466
Name: count, dtype: int64


## 3. User-Level Event Deduplication

Remove repeated tracking events for the same user at the
same funnel stage.

Each user should contribute at most one record per stage.

In [4]:
# Check whether users have consistent device assignments
device_consistency = df.groupby('user_id')['device'].nunique()

print(
    "Users with multiple device assignments:",
    (device_consistency > 1).sum()
)

# Deduplicate by user and funnel event
df_dedup = df.drop_duplicates(
    subset=['user_id', 'event_name']
).copy()

# Verify deduplication
print("\nOriginal rows:", len(df))
print("Deduplicated rows:", len(df_dedup))
print("Duplicate records removed:", len(df) - len(df_dedup))

# Verify no repeated user-stage combinations remain
print(
    "Remaining user-stage duplicates:",
    df_dedup.duplicated(
        subset=['user_id', 'event_name']
    ).sum()
)
print("Original dataset shape:", df.shape)
print("Deduplicated dataset shape:", df_dedup.shape)

Users with multiple device assignments: 0

Original rows: 26061
Deduplicated rows: 23061
Duplicate records removed: 3000
Remaining user-stage duplicates: 0
Original dataset shape: (26061, 3)
Deduplicated dataset shape: (23061, 3)


## 4. Funnel Stage Definition

Define the five funnel stages in their intended business sequence.
This ordering will be used consistently throughout the analysis.

In [5]:
funnel_steps = [
    '1_Homepage',
    '2_Product_View',
    '3_Add_to_Cart',
    '4_Checkout_Start',
    '5_Signup_Complete'
]

print("Funnel stages:")
for step in funnel_steps:
    print(step)

Funnel stages:
1_Homepage
2_Product_View
3_Add_to_Cart
4_Checkout_Start
5_Signup_Complete


## 5. Funnel Metrics Calculation

With the clickstream data cleaned and duplicate user-stage records removed, the next step is to quantify user progression through the conversion funnel.

The analysis measures:

- **Unique Users:** Distinct users reaching each funnel stage.
- **Absolute Drop-off:** Number of users lost between consecutive stages.
- **Step-over-Step Conversion Rate:** Percentage of users progressing from the previous stage to the current stage.
- **Step Drop-off Rate:** Percentage of users lost between consecutive stages.
- **Cumulative Conversion Rate:** Percentage of homepage users reaching each subsequent stage.

These metrics will establish the overall conversion baseline and identify the stages with the greatest user attrition.

In [6]:
# Calculate unique users reaching each funnel stage
funnel_users = (
    df_dedup.groupby('event_name')['user_id']
    .nunique()
    .reindex(funnel_steps, fill_value=0)
)

# Display funnel user counts
print("Unique users at each funnel stage:")
display(funnel_users)

Unique users at each funnel stage:


event_name
1_Homepage           10000
2_Product_View        6731
3_Add_to_Cart         3396
4_Checkout_Start      1633
5_Signup_Complete     1301
Name: user_id, dtype: int64

In [7]:
# Create a funnel metrics DataFrame
funnel_metrics = pd.DataFrame({
    'users': funnel_users
})

# Calculate users lost between consecutive stages
funnel_metrics['users_lost'] = (
    funnel_metrics['users'].shift(1) - funnel_metrics['users']
)

# Calculate step-over-step conversion rate
funnel_metrics['step_conversion_rate'] = (
    funnel_metrics['users'] / funnel_metrics['users'].shift(1) * 100
)

# Calculate step drop-off rate
funnel_metrics['step_dropoff_rate'] = (
    100 - funnel_metrics['step_conversion_rate']
)

# Calculate cumulative conversion rate
funnel_metrics['cumulative_conversion_rate'] = (
    funnel_metrics['users'] / funnel_metrics['users'].iloc[0] * 100
)

# Homepage is the baseline; transition metrics are not applicable
funnel_metrics.iloc[0, 1:4] = np.nan
funnel_metrics.iloc[0, 4] = 100

# Display funnel metrics
display(funnel_metrics.round(2))

,users,users_lost,step_conversion_rate,step_dropoff_rate,cumulative_conversion_rate
event_name,,,,,
1_Homepage,10000,NaN,NaN,NaN,100.00
2_Product_View,6731,3269.0,67.31,32.69,67.31
3_Add_to_Cart,3396,3335.0,50.45,49.55,33.96
4_Checkout_Start,1633,1763.0,48.09,51.91,16.33
5_Signup_Complete,1301,332.0,79.67,20.33,13.01


In [8]:
# Calculate unique users by device and funnel stage
device_funnel_users = (
    df_dedup.groupby(['device', 'event_name'])['user_id']
    .nunique()
    .unstack(level='event_name')
    .reindex(columns=funnel_steps, fill_value=0)
)

# Display device-level funnel counts
display(device_funnel_users)

event_name,1_Homepage,2_Product_View,3_Add_to_Cart,4_Checkout_Start,5_Signup_Complete
device,,,,,
desktop,5004,3487,1966,1202,964
mobile,4996,3244,1430,431,337


In [9]:
# Calculate step-over-step conversion rate by device
device_step_cr = (
    device_funnel_users.div(
        device_funnel_users.shift(axis=1)
    ) * 100
)

# Calculate step drop-off rate by device
device_step_dropoff = 100 - device_step_cr

# Calculate cumulative conversion rate by device
device_cumulative_cr = (
    device_funnel_users.div(
        device_funnel_users.iloc[:, 0],
        axis=0
    ) * 100
)

# Display device-level conversion rates
print("Step-over-step Conversion Rate (%)")
display(device_step_cr.round(2))

print("Step Drop-off Rate (%)")
display(device_step_dropoff.round(2))

print("Cumulative Conversion Rate (%)")
display(device_cumulative_cr.round(2))

Step-over-step Conversion Rate (%)


event_name,1_Homepage,2_Product_View,3_Add_to_Cart,4_Checkout_Start,5_Signup_Complete
device,,,,,
desktop,NaN,69.68,56.38,61.14,80.20
mobile,NaN,64.93,44.08,30.14,78.19


Step Drop-off Rate (%)


event_name,1_Homepage,2_Product_View,3_Add_to_Cart,4_Checkout_Start,5_Signup_Complete
device,,,,,
desktop,NaN,30.32,43.62,38.86,19.80
mobile,NaN,35.07,55.92,69.86,21.81


Cumulative Conversion Rate (%)


event_name,1_Homepage,2_Product_View,3_Add_to_Cart,4_Checkout_Start,5_Signup_Complete
device,,,,,
desktop,100.0,69.68,39.29,24.02,19.26
mobile,100.0,64.93,28.62,8.63,6.75


In [10]:
# Combine device-level funnel metrics
device_funnel_metrics = pd.concat(
    {
        'users': device_funnel_users,
        'step_conversion_rate': device_step_cr,
        'step_dropoff_rate': device_step_dropoff,
        'cumulative_conversion_rate': device_cumulative_cr
    },
    axis=1
)

# Display consolidated device funnel metrics
display(device_funnel_metrics.round(2))

users                                                \
event_name 1_Homepage 2_Product_View 3_Add_to_Cart 4_Checkout_Start   
device                                                                
desktop          5004           3487          1966             1202   
mobile           4996           3244          1430              431   

                             step_conversion_rate                 \
event_name 5_Signup_Complete           1_Homepage 2_Product_View   
device                                                             
desktop                  964                  NaN          69.68   
mobile                   337                  NaN          64.93   

                                                            step_dropoff_rate  \
event_name 3_Add_to_Cart 4_Checkout_Start 5_Signup_Complete        1_Homepage   
device                                                                          
desktop            56.38            61.14             80.20               NaN   
mobile             44.08            30.14             78.19               NaN   

                                                                            \
event_name 2_Product_View 3_Add_to_Cart 4_Checkout_Start 5_Signup_Complete   
device                                                                       
desktop             30.32         43.62            38.86             19.80   
mobile              35.07         55.92            69.86             21.81   

           cumulative_conversion_rate                               \
event_name                 1_Homepage 2_Product_View 3_Add_to_Cart   
device                                                               
desktop                         100.0          69.68         39.29   
mobile                          100.0          64.93         28.62   

                                               
event_name 4_Checkout_Start 5_Signup_Complete  
device                                         
desktop               24.02             19.26  
mobile                 8.63              6.75

In [11]:
import plotly.graph_objects as go
from pathlib import Path

# Create funnel visualization
fig = go.Figure(
    go.Funnel(
        y=funnel_steps,
        x=funnel_users.values,
        textinfo="value+percent initial+percent previous",
        marker=dict(
            color=["#4472C4", "#5B9BD5", "#70AD47", "#ED7D31", "#C00000"]
        )
    )
)

# Customize chart layout
fig.update_layout(
    title="Overall Conversion Funnel",
    title_x=0.5,
    funnelmode="stack",
    template="plotly_white",
    height=600,
    width=1000,
    xaxis_title="Unique Users",
    yaxis_title="Funnel Stage"
)

# Display visualization
fig.show()

# Save visualization as HTML
output_dir = Path.cwd()

if output_dir.name == "notebooks":
    output_dir = output_dir.parent

output_dir = output_dir / "findings"
output_dir.mkdir(exist_ok=True)

fig.write_html(output_dir / "02_overall_conversion_funnel.html")

print("Funnel visualization saved successfully.")

Funnel visualization saved successfully.


In [13]:
import plotly.graph_objects as go

fig = go.Figure()

for device in device_funnel_users.index:
    fig.add_trace(
        go.Funnel(
            name=device.capitalize(),
            y=funnel_steps,
            x=device_funnel_users.loc[device].values,
            textinfo="value+percent initial",
        )
    )

fig.update_layout(
    title="Conversion Funnel: Desktop vs Mobile",
    title_x=0.5,
    funnelmode="group",
    template="plotly_white",
    height=600,
    width=1100,
    xaxis_title="Unique Users",
    yaxis_title="Funnel Stage",
    legend_title="Device"
)

fig.show()

In [14]:
from pathlib import Path

output_dir = Path.cwd()
if output_dir.name == "notebooks":
    output_dir = output_dir.parent

output_dir = output_dir / "findings"
output_dir.mkdir(exist_ok=True)

fig.write_html(output_dir / "02_device_conversion_funnel.html")

print("Device funnel visualization saved:", 
      (output_dir / "02_device_conversion_funnel.html").exists())

Device funnel visualization saved: True
